<a href="https://colab.research.google.com/github/E-Sentinel-Project/E-Sentinel/blob/master/BatteryStats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

# Upload file
uploaded = files.upload()

# Get uploaded file name
file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)



Saving batterystats.txt to batterystats.txt
Uploaded file: batterystats.txt


In [4]:
with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
    content = f.read()

print("File loaded successfully.")


File loaded successfully.


In [6]:
from google.colab import files

uploaded = files.upload()

# Get file name
file_name = list(uploaded.keys())[0]

print("Uploaded:", file_name)


Saving batterystats.txt to batterystats (1).txt
Uploaded: batterystats (1).txt


In [7]:
import re

APP_PACKAGE = "com.example.e_sentinel"

# Read uploaded file
with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
    content = f.read()

# Regex patterns
time_pattern = re.compile(r"\+(\d+)m(\d+)s(\d+)ms")
charge_pattern = re.compile(r"charge=(\d+)")

timestamps = []
charges = []
inside_app_window = False

for line in content.splitlines():

    # Detect foreground start
    if '+top=' in line and APP_PACKAGE in line:
        inside_app_window = True

    # Detect foreground end
    if '-top=' in line and APP_PACKAGE in line:
        inside_app_window = False

    # Collect data only when app active
    if inside_app_window:
        t_match = time_pattern.search(line)
        c_match = charge_pattern.search(line)

        if t_match and c_match:
            minutes = int(t_match.group(1))
            seconds = int(t_match.group(2))
            millis  = int(t_match.group(3))

            total_seconds = minutes * 60 + seconds + millis / 1000
            charge = int(c_match.group(1))

            timestamps.append(total_seconds)
            charges.append(charge)

# Results
if len(charges) < 2:
    print("Not enough E-Sentinel battery data found.")
else:
    initial_charge = charges[0]
    final_charge = charges[-1]

    battery_used = final_charge - initial_charge
    total_time_hr = (timestamps[-1] - timestamps[0]) / 3600
    avg_consumption = battery_used / total_time_hr if total_time_hr > 0 else 0

    print("----- E-Sentinel Battery Consumption -----")
    print(f"App Package         : {APP_PACKAGE}")
    print(f"Battery Used        : {battery_used:.2f} mAh")
    print(f"Active Time         : {total_time_hr:.2f} hours")
    print(f"Average Consumption : {avg_consumption:.2f} mAh/hour")


----- E-Sentinel Battery Consumption -----
App Package         : com.example.e_sentinel
Battery Used        : 171.00 mAh
Active Time         : 0.79 hours
Average Consumption : 215.51 mAh/hour
